**Agenda**

Um agente pode não travar nunca e mesmo assim piorar: um ajuste que parecia corrigir um problema e na
verdade regrediu outro, uma taxa de sucesso caindo devagar, uma execução específica que ninguém sabe
explicar. Guardrail e human-in-the-loop (Aula 4) não pegam nada disso.

- três perguntas, três momentos: antes de subir uma mudança, ao longo de muitas execuções, e dentro de uma
  execução específica;
- evals, monitoring e tracing como resposta a cada uma;
- como as três se conectam num único ciclo de melhoria contínua.

# [Conceito] Verificação e observabilidade de agentes

## Um agente que nunca trava

O assistente da Clínica Alura está no ar há três semanas, com guardrails e aprovação humana da Aula 4
funcionando. Nenhum erro no terminal, nenhum guardrail disparando.

Mesmo assim, três coisas aconteceram essa semana:

- alguém ajustou a instrução de classificação pra reduzir um tipo de erro, e ninguém testou contra os
  outros casos que já funcionavam;
- a recepção comenta que o assistente parece mais lento que no lançamento, mas ninguém mediu;
- numa conversa específica, ninguém sabe dizer se o agente consultou os dados da clínica antes de
  responder, ou só chutou algo plausível.

Nenhuma dessas coisas é do tipo que um guardrail bloqueia. O agente continua rodando, e continua piorando de
um jeito que ninguém consegue ver.

## Três momentos

Essas três situações são a mesma pergunta, feita em três momentos diferentes: essa versão é melhor que a
anterior, o comportamento mudou, o que aconteceu aqui.

<img src="resources/verification_timeline.png" width="100%">

## Evals: essa mudança pode ir pra produção?

Um teste é determinístico: a mesma entrada sempre deveria dar a mesma saída, certa ou errada. Um agente não
funciona assim, a mesma pergunta pode ser respondida de formas diferentes e ainda estar correta.

Uma avaliação mede isso de outro jeito: reúne um conjunto de cenários representativos (um golden dataset),
roda a versão candidata contra eles, e compara a taxa de sucesso com a versão anterior. Só sobe pra produção
se não regredir.

É uma verificação offline: acontece antes de qualquer paciente de verdade ver a resposta.

## Monitoring: o comportamento mudou?

Monitoring é detecção. Ele observa muitas execuções ao longo do tempo (taxa de sucesso, custo, latência,
chamadas de tool, aprovações humanas) e aponta quando algo sai do padrão. Descobrir o motivo vem depois, com
tracing.

Uma execução lenta é ruído. Uma semana inteira de execuções lentas é sinal. Sem agregar as execuções, cada
uma passa e some, e ninguém enxerga a tendência.

## Tracing: o que aconteceu aqui, exatamente?

Quando o monitoring aponta um problema, ou alguém reclama de uma conversa específica, a pergunta muda de
"quantas" pra "essa aqui, o que exatamente aconteceu".

Um trace é a árvore de passos de uma execução: cada chamada ao modelo, cada chamada de tool, cada decisão,
na ordem em que aconteceram. Ele não aponta a causa raiz sozinho, mas mostra o caminho que o agente
realmente seguiu, a peça que faltava pra reconstruir o raciocínio por trás de uma resposta.

## Como as três se conectam

As três formam um único ciclo de melhoria contínua. Um trace não serve só pra explicar um caso isolado:
monitoring é, na prática, agregação de muitos traces ao longo do tempo. Quando um trace explica uma falha,
essa falha vira um cenário novo no dataset de evals; a próxima versão só sobe se passar nesse cenário
também, o que a leva de volta pro início do ciclo: uma versão nova, rastreada e monitorada como todas as
outras.

<img src="resources/feedback_loop.png" width="90%">

## As três, lado a lado

| | Evals | Monitoring | Tracing |
|---|---|---|---|
| Quando roda | Antes de subir uma mudança (offline) | Continuamente, em produção | Sob demanda, numa execução específica |
| Granularidade | Um conjunto de cenários | Muitas execuções agregadas | Uma execução, passo a passo |
| Pergunta que responde | Essa versão é melhor que a anterior? | O comportamento mudou? | O que aconteceu, exatamente? |
| Fonte de dado | Golden dataset (cenários conhecidos) | Métricas agregadas de várias execuções | Spans de uma execução |

## Isso não nasceu com IA

Software clássico já observa produção com métricas e traces, os dois pilares mais próximos do que vimos
aqui: métricas agregam sinais de muitas execuções, traces mostram o caminho de uma execução específica. A
pergunta que eles respondem, historicamente, é se o sistema está no ar.

Um agente pode estar perfeitamente no ar e ainda estar errando: nenhum dos três exemplos do começo derrubou
o servidor. Monitoring e tracing são essa mesma ideia, adaptada pra observar comportamento, não só
disponibilidade. Evals não tem equivalente clássico: é a peça nova, porque só faz sentido quando a mesma
entrada pode gerar respostas diferentes e ainda estar certa.

## Nesta aula

1. Evals: teste vs avaliação.
2. Monitoring: detectar sem diagnosticar.
3. Tracing com Langfuse.
4. Projeto: três desafios pra continuar sozinho.